# Multiple Linear Regression and Feature Scaling
Predicting with many inputs at once, and why scaling the inputs matters. Runs free in Google Colab.

Run each cell from top to bottom with **Shift + Enter**.

## 1. The data
A used car dealer wants to price cars from four facts: age, kilometres driven, engine size and transmission.

### 200 used cars
- `used_cars.csv`: 200 cars: age in years, kilometres in thousands, engine size in litres, transmission, and price in dollars

In [ ]:
import pandas as pd

url = ("https://raw.githubusercontent.com/Uttam-Nepal/"
       "machine-learning-explained/main/"
       "multiple-regression-and-scaling/used_cars.csv")
df = pd.read_csv(url)
print(df.shape)
df.head()

### Turn text into a number
- `(df["transmission"] == "automatic")`: True for automatic cars
- `.astype(int)`: True becomes 1, False becomes 0
- `X and y`: four input columns, and the price to predict

In [ ]:
df["automatic"] = (df["transmission"] == "automatic").astype(int)
features = ["age_years", "km_thousands", "engine_litres", "automatic"]
X = df[features]
y = df["price"]
print(X.head(3))

## 2. Multiple linear regression
**price = b + w1 x age + w2 x km + w3 x engine + w4 x automatic**. Each weight is the change in price when that input goes up by 1 and the others stay the same.

### Fit the model
- `train_test_split`: keep a quarter of the cars hidden, for a fair test
- `model.coef_`: one weight per input, in the same order as features
- `age_years`: about -1,600: each extra year costs about $1,600, if everything else stays the same
- `km_thousands`: about -27: each extra 1,000 km costs about $27
- `automatic`: an automatic is worth about $1,800 more

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=1)
model = LinearRegression().fit(X_train, y_train)
print("intercept:", round(model.intercept_))
for name, w in zip(features, model.coef_):
    print(f"{name:>14}: {w:8.1f}")

### How good is it?
- `r2_score`: 1 is perfect, 0 is no better than guessing the average price
- `mean_absolute_error`: the average miss in dollars, on cars the model never saw

In [ ]:
from sklearn.metrics import r2_score, mean_absolute_error

pred = model.predict(X_test)
print("R squared:", round(r2_score(y_test, pred), 3))
print("typical miss: $", round(mean_absolute_error(y_test, pred)))

### Compare with a single input
- `X_train[["age_years"]]`: the same model with just one input
- `the result`: more inputs explain more of the price

In [ ]:
one = LinearRegression().fit(X_train[["age_years"]], y_train)
pred_one = one.predict(X_test[["age_years"]])
print("age only, R squared:", round(r2_score(y_test, pred_one), 3))

## 3. Why the weights can't be compared directly
Each weight depends on its input's units. **Standardising** turns every input into 'how many standard deviations above or below average': z = (x - mean) / standard deviation.

### Very different scales
- `mean`: the average of each column
- `std`: the standard deviation: the typical distance from the average
- `the point`: km goes up to over 200, engine size only up to 3

In [ ]:
print(X_train.describe().loc[["mean", "std", "min", "max"]].round(1))

### Standardise the inputs
- `StandardScaler().fit(X_train)`: learn each column's mean and standard deviation, from the training cars only
- `scaler.transform`: apply (x - mean) / std to each column, in both sets
- `the output`: every column now has mean 0 and standard deviation 1

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)
X_test_s = scaler.transform(X_test)
print(pd.DataFrame(X_train_s, columns=features).describe()
      .loc[["mean", "std"]].round(2))

### Weights on a common scale
- `model_s`: the same model, trained on standardised inputs
- `each weight`: the price change for one standard deviation more of that input
- `the output`: now the sizes can be compared: age and km matter most

In [ ]:
model_s = LinearRegression().fit(X_train_s, y_train)
for name, w in zip(features, model_s.coef_):
    print(f"{name:>14}: {w:8.0f}")

## 4. Scaling and gradient descent
Models trained by gradient descent need scaled inputs. **SGDRegressor** is linear regression trained by stochastic gradient descent.

### Without and with scaling
- `SGDRegressor`: a linear model that learns by walking downhill on the error
- `raw inputs`: the steps are thrown around by the big km numbers: an R squared of minus a billion billion
- `scaled inputs`: the same model works perfectly

In [ ]:
from sklearn.linear_model import SGDRegressor

raw = SGDRegressor(random_state=0).fit(X_train, y_train)
r_raw = r2_score(y_test, raw.predict(X_test))
print(f"raw inputs, R squared:    {r_raw:.2g}")

scaled = SGDRegressor(random_state=0).fit(X_train_s, y_train)
r_s = r2_score(y_test, scaled.predict(X_test_s))
print(f"scaled inputs, R squared: {r_s:.3f}")

### A pipeline: scale and fit in one step
- `make_pipeline`: chain the scaler and the model together
- `pipe.fit`: learns the scaling from the training data, then trains the model
- `pipe.score`: R squared on the test cars; the scaling is applied automatically

In [ ]:
from sklearn.pipeline import make_pipeline

pipe = make_pipeline(StandardScaler(), SGDRegressor(random_state=0))
pipe.fit(X_train, y_train)
print("pipeline, R squared:", round(pipe.score(X_test, y_test), 3))

## 5. Mini project: a price estimator
Price any car with the trained model.

### Price a car
- `pd.DataFrame([[...]], columns=features)`: one car, as a table with the same columns
- `round(..., -2)`: round to the nearest $100, like a real price tag

In [ ]:
def price_car(age, km, engine, automatic):
    car = pd.DataFrame([[age, km, engine, automatic]], columns=features)
    return int(round(model.predict(car)[0], -2))

print("3 years, 40k km, 1.6 L, auto:  $", price_car(3, 40, 1.6, 1))
print("12 years, 180k km, 1.2 L, manual: $", price_car(12, 180, 1.2, 0))

## Try it yourself
1. Price a 2-year-old manual car with 25,000 km and a 1.4 litre engine.
2. Fit the model with only age and km. How much does the test R squared drop?
3. Try MinMaxScaler instead of StandardScaler in the pipeline. Do the predictions change?